In [3]:
!ls

sample_data


In [1]:
!nvidia-smi

Tue Jan 27 08:36:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("Cornell-University/arxiv")

print("Path to dataset files:", path)

100%|██████████| 1.56G/1.56G [00:12<00:00, 134MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/Cornell-University/arxiv/versions/270


In [2]:
import json
import os 
file_path = os.path.join(path, "arxiv-metadata-oai-snapshot.json")

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        paper = json.loads(line)
        print(paper.keys())
        print(paper["title"])
        break  # remove this to read all


dict_keys(['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed'])
Calculation of prompt diphoton production cross sections at Tevatron and
  LHC energies


In [5]:
import pandas as pd
import json

rows = []
with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        rows.append(json.loads(line))
        if i == 200:  # limit size
            break

df = pd.DataFrame(rows)
df.head()


,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
0,0704.0001,Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",Calculation of prompt diphoton production cros...,"37 pages, 15 figures; published version","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,ANL-HEP-PR-07-12,hep-ph,None,A fully differential calculation in perturba...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2008-11-26,"[[Balázs, C., ], [Berger, E. L., ], [Nadolsky,..."
1,0704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,None,None,None,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib...,"We describe a new algorithm, the $(k,\ell)$-...","[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]"
2,0704.0003,Hongjun Pan,Hongjun Pan,The evolution of the Earth-Moon system based o...,"23 pages, 3 figures",None,None,None,physics.gen-ph,None,The evolution of Earth-Moon system is descri...,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...",2008-01-13,"[[Pan, Hongjun, ]]"
3,0704.0004,David Callan,David Callan,A determinant of Stirling cycle numbers counts...,11 pages,None,None,None,math.CO,None,We show that a determinant of Stirling cycle...,"[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2007-05-23,"[[Callan, David, ]]"
4,0704.0005,Alberto Torchinsky,Wael Abu-Shammala and Alberto Torchinsky,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,None,"Illinois J. Math. 52 (2008) no.2, 681-689",None,None,math.CA math.FA,None,In this paper we show how to compute the $\L...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2013-10-15,"[[Abu-Shammala, Wael, ], [Torchinsky, Alberto, ]]"


In [4]:
!pip install numpy pandas tqdm sentence-transformers optuna faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 115.6 MB/s eta 0:00:0000:0100:01


In [8]:
import os, json
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


@dataclass
class Paper:
    paper_id: str
    title: str
    abstract: str


def load_arxiv_jsonl(jsonl_path: str, limit: Optional[int] = None) -> List[Paper]:
    papers = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            pid = obj.get("id", "")
            title = (obj.get("title") or "").strip().replace("\n", " ")
            abstract = (obj.get("abstract") or "").strip().replace("\n", " ")
            if pid and title:
                papers.append(Paper(pid, title, abstract))
            if limit is not None and len(papers) >= limit:
                break
    return papers


def l2_normalize(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.clip(n, eps, None)


def build_ip_index(vectors: np.ndarray) -> faiss.Index:
    d = vectors.shape[1]
    idx = faiss.IndexFlatIP(d)
    idx.add(vectors.astype(np.float32))
    return idx


class HierarchicalSearch:
    """
    Stage 1: Title-only FAISS retrieval
    Stage 2: Rerank topN using abstract (either precomputed or computed on-the-fly)

    Score:
      score = w_title*cos(q, title) + w_abs*cos(q, abstract)
    """
    def __init__(
        self,
        model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
        device: str = "cpu",
        batch_size: int = 128,
        use_precomputed_abstracts: bool = False,
    ):
        self.model = SentenceTransformer(model_name, device=device)
        self.batch_size = batch_size
        self.use_precomputed_abstracts = use_precomputed_abstracts

        self.papers: List[Paper] = []
        self.title_vecs: Optional[np.ndarray] = None
        self.abs_vecs: Optional[np.ndarray] = None

        self.title_index: Optional[faiss.Index] = None

    def fit(self, papers: List[Paper]) -> None:
        self.papers = papers

        titles = [p.title for p in papers]
        title_vecs = self.model.encode(
            titles, batch_size=self.batch_size, show_progress_bar=True, convert_to_numpy=True
        )
        title_vecs = l2_normalize(title_vecs).astype(np.float32)
        self.title_vecs = title_vecs
        self.title_index = build_ip_index(self.title_vecs)

        if self.use_precomputed_abstracts:
            abstracts = [p.abstract if p.abstract else "" for p in papers]
            abs_vecs = self.model.encode(
                abstracts, batch_size=self.batch_size, show_progress_bar=True, convert_to_numpy=True
            )
            abs_vecs = l2_normalize(abs_vecs).astype(np.float32)
            self.abs_vecs = abs_vecs

    def search(
        self,
        query: str,
        top_k: int = 10,
        stage1_candidates: int = 100,
        w_title: float = 0.4,
        w_abs: float = 0.6,
    ) -> List[Tuple[Paper, float]]:
        if self.title_index is None or self.title_vecs is None:
            raise RuntimeError("Call fit() first.")

        # Embed query once
        q = self.model.encode([query], convert_to_numpy=True)
        q = l2_normalize(q).astype(np.float32)

        # ---- Stage 1: title retrieval
        scores1, ids1 = self.title_index.search(q, stage1_candidates)
        cand_ids = ids1[0]
        cand_ids = [i for i in cand_ids.tolist() if i != -1]

        # ---- Stage 2: rerank using weighted title+abstract similarity
        results = []
        for idx in cand_ids:
            s_title = float(np.dot(q[0], self.title_vecs[idx]))

            # abstract similarity:
            if self.use_precomputed_abstracts:
                if self.abs_vecs is None:
                    raise RuntimeError("Abstract vectors not computed; set use_precomputed_abstracts=True in fit.")
                s_abs = float(np.dot(q[0], self.abs_vecs[idx]))
            else:
                # Compute abstract embedding ONLY for these candidates
                abs_text = self.papers[idx].abstract or ""
                abs_vec = self.model.encode([abs_text], convert_to_numpy=True)
                abs_vec = l2_normalize(abs_vec).astype(np.float32)
                s_abs = float(np.dot(q[0], abs_vec[0]))

            score = w_title * s_title + w_abs * s_abs
            results.append((idx, score))

        results.sort(key=lambda x: x[1], reverse=True)
        results = results[:top_k]
        return [(self.papers[i], s) for i, s in results]


# -------------------------
# Example usage
# -------------------------
if __name__ == "__main__":
    jsonl_path = os.path.join(path, "arxiv-metadata-oai-snapshot.json")
    # Load some papers (start small while iterating)
    papers = load_arxiv_jsonl(jsonl_path, limit=20000) 

    # Option A: precompute abstracts (fast query time)
    engine = HierarchicalSearch(use_precomputed_abstracts=False, device="cuda")
    engine.fit(papers)

    # Option B: compute abstracts on demand (slow query time, no precompute)
    # engine = HierarchicalSearch(use_precomputed_abstracts=False, device="cpu")
    # engine.fit(papers)

    query = "retrieval augmented generation for scientific literature"
    hits = engine.search(query, top_k=10, stage1_candidates=100, w_title=0.3, w_abs=0.7)

    for p, s in hits:
        print(f"{s:.4f} | {p.paper_id} | {p.title[:120]}")


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

0.4856 | 0705.0751 | Approximate textual retrieval
0.4818 | 0704.2963 | Using Access Data for Paper Recommendations on ArXiv.org
0.4202 | 0707.1913 | Removing Manually-Generated Boilerplate from Electronic Texts:   Experiments with Project Gutenberg e-Books
0.4174 | 0708.1150 | A Practical Ontology for the Large-Scale Modeling of Scholarly Artifacts   and their Usage
0.3993 | 0707.3696 | Towards journalometrical analysis of a scientific periodical: a case   study
0.3891 | 0707.0745 | Semantic Information Retrieval from Distributed Heterogeneous Data   Sources
0.3788 | 0705.2106 | Scientific citations in Wikipedia
0.3761 | 0707.3261 | Scaling rules in the science system: influence of field-specific   citation characteristics on the impact of research gr
0.3723 | 0707.3575 | An exploratory study of Google Scholar
0.3692 | 0705.4606 | Dynamic User-Defined Similarity Searching in Semi-Structured Text   Retrieval
